# 06 — Análise exploratória
Distribuição de classes, confundidores (site, idade, sexo) e padrão médio de conectividade.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import config

df = pd.read_csv(config.ARQ_PARTICIPANTES, dtype={"ID_padronizado": str})
fc = config.carregar_fc(df["ID_padronizado"])
y = df["classe"].values
print("FC:", fc.shape, "| classes:", np.bincount(y))

In [ ]:
print("Classe por site:")
tab = pd.crosstab(df["Site"], df["classe"])
display(tab)
chi2, p, *_ = stats.chi2_contingency(tab)
print(f"Qui-quadrado site x classe: chi2={chi2:.1f}, p={p:.2e}")

tab.plot(kind="bar", stacked=True, figsize=(7, 4), title="Classe por site")
plt.tight_layout(); plt.savefig(config.FIGURAS / "06_classe_por_site.png", dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(data=df, x="Age", hue="classe", kde=True, ax=ax[0], stat="density", common_norm=False)
sns.countplot(data=df, x="Gender", hue="classe", ax=ax[1])
plt.tight_layout(); plt.savefig(config.FIGURAS / "06_idade_sexo.png", dpi=150); plt.show()

a0, a1 = df.loc[y == 0, "Age"].dropna(), df.loc[y == 1, "Age"].dropna()
print("Idade controle vs TDAH (Mann-Whitney):", stats.mannwhitneyu(a0, a1))

In [ ]:
iu = np.triu_indices(config.N_ROIS, k=1)
fc_media_sujeito = fc[:, iu[0], iu[1]].mean(axis=1)
df["fc_media"] = fc_media_sujeito

plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="Site", y="fc_media", hue="classe")
plt.title("Conectividade média por site e classe")
plt.tight_layout(); plt.savefig(config.FIGURAS / "06_fc_media_site.png", dpi=150); plt.show()

In [ ]:
m0, m1 = fc[y == 0].mean(0), fc[y == 1].mean(0)
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
for a, m, t in zip(ax, (m0, m1, m1 - m0), ("Controle", "TDAH", "TDAH − Controle")):
    lim = 0.15 if "−" in t else 0.8
    im = a.imshow(m, cmap="RdBu_r", vmin=-lim, vmax=lim)
    a.set_title(t); plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.savefig(config.FIGURAS / "06_fc_media_classes.png", dpi=150); plt.show()

**Atenção:** se `site` estiver fortemente associado à classe (qui-quadrado acima), um modelo pode aprender o site em vez do diagnóstico. Considere isso na interpretação dos resultados dos notebooks 08–11.